In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

In [2]:
df = pd.read_csv("../../data/processed/diabetic_retinopathy/image_metadata.csv")

In [3]:
df

,id_code,height,width,diagnosis,file_path,blur_score,brightness
0,0024cdab0c1e,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0024c...,262.95,66.78
1,00cb6555d108,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\00cb6...,238.39,65.31
2,0124dffecf29,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0124d...,406.80,90.57
3,01b3aed3ed4c,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\01b3a...,252.08,77.22
4,0369f3efe69b,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0369f...,287.19,59.32
...,...,...,...,...,...,...,...
3657,f9156aeffc5e,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\f91...,135.47,34.86
3658,fb61230b99dd,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fb6...,133.34,54.09
3659,fcc6aa6755e6,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fcc...,148.21,37.61
3660,fda39982a810,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fda...,115.17,45.88


# train/validation/test split

In [4]:
from sklearn.model_selection import train_test_split

# train/test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["diagnosis"],
    random_state=10
)
# train/validation split
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["diagnosis"],
    random_state=10
)

len(train_df), len(val_df), len(test_df)

(2343, 586, 733)

In [5]:
len(train_df) + len(val_df) + len(test_df)

3662

### Percentage of each part

In [6]:
print((len(train_df) * 100) / 3662)
print((len(val_df) * 100) / 3662)
print((len(test_df) * 100) / 3662)

63.981430912069904
16.002184598580012
20.01638448935008


#### Checking torch version

In [7]:
# !pip install torch torchvision

In [8]:
import torch
torch.__version__

'2.13.0+cpu'

## Set device type

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [10]:
# X_train, X_test = X_train, X_test
# y_train, y_test = y_train, y_test

## Build a class for dataset

In [11]:
from torch.utils.data import Dataset

class RetinopathyDataset(Dataset):
    
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        
        image = cv2.imread(row["file_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        label = row["diagnosis"]
        
        if self.transform:
            image = self.transform(image)

        return image, label
    

## Set transforms

#### Calculate Mean/Std RGB for train_df

##### import module

In [12]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]

sys.path.append(str(ROOT))

In [13]:
from shared.preprocessing.statistics import get_mean_std_rgb

mean, std = get_mean_std_rgb(train_df)
mean, std

(array([0.41221823, 0.21995935, 0.07291832]),
 array([0.27404434, 0.1498604 , 0.0806435 ]))

###### Mean RGB for the whole dataset: [0.41346971 0.22068593 0.07336238]
###### Std RGB for the whole dataset: [0.27443648 0.14982664 0.08072127]

In [14]:
from torchvision import transforms


train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

## Create dataset

In [15]:
train_dataset = RetinopathyDataset(
    dataframe=train_df,
    transform=train_transform,
)

val_dataset = RetinopathyDataset(
    dataframe=val_df,
    transform=val_transform,
)

test_dataset = RetinopathyDataset(
    dataframe=test_df,
    transform=val_transform,
)


## Create Dataloader

##### Logical Processors

In [16]:
import os

print(os.cpu_count())

12


In [17]:
from torch.utils.data import DataLoader

num_workers = os.cpu_count()
pin_memory = torch.cuda.is_available()


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)



In [18]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(images.dtype)
print(labels.dtype)

torch.Size([32, 3, 224, 224])
torch.Size([32])
torch.float32
torch.int64
